# Геолокация: финальный эксперимент

Два замороженных эмбеддинга (GeoCLIP 512 + StreetCLIP 768) → географические вероятности, поиск по похожим снимкам, прямая регрессия и стекинг. 50 конфигураций V4, ещё 500 вариантов скрининга, 10 адаптаций из статей и полноразмерное сравнение финалистов. Победитель выбирается только внутри 212 500 снимков разработки по MSE расстояний WGS84.

![Схема обучения](docs/workflow-final500.svg)

[Полный отчёт по каждому эксперименту](docs/report-500.md) · [Архитектуры](docs/architecture-final.md) · [Исследованные статьи](docs/literature-notes.md)

**Измеренный итог:** `tuned:all/shared/expert/a0.01/tail0/cap20000`. CV RMSE 973.49 км, тест MAE 271.85 км, тест RMSE 952.52 км. Тест — 37 500 снимков; параметры выбраны по CV. [Схема победителя](docs/winner-architecture.svg).

## 1. Настройка среды и фиксированное разбиение

Выберите GPU Colab (эксперимент выполнен на T4). Запустите из распакованной папки проекта или задайте путь к ZIP. Укажите папку исходных данных. Отдельно сохраняются 37 500 тестовых строк, остальные 212 500 составляют пять фолдов по 42 500. Для запуска с нуля не используются исторические тестовые метрики как цель оптимизации.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from IPython.display import SVG, display

REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
TRAIN_250K_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
GEO_CV_ROOT = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
here = Path.cwd()
repo = next((p for p in (checkout, here, here.parent) if (p / "src" / "screen" / "full_search.py").is_file()), None)
if repo is None:
    if not ZIP_PATH:
        raise FileNotFoundError("Укажите ZIP_PATH или путь к новой версии проекта в REPO_DIR.")
    if checkout.exists():
        raise FileExistsError("Для нового ZIP укажите свободный путь в REPO_DIR.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        roots = [Path(tmp), *Path(tmp).iterdir()]
        source = next((p for p in roots if (p / "src" / "screen" / "full_search.py").is_file()), None)
        if source is None:
            raise FileNotFoundError("В ZIP отсутствует src/screen/full_search.py.")
        from shutil import copytree
        copytree(source, checkout)
    repo = checkout

data, work = (Path(p).expanduser().resolve() for p in (TRAIN_250K_DIR, GEO_CV_ROOT))
env = {**os.environ, "TRAIN_250K_DIR": str(data), "GEO_CV_ROOT": str(work)}
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите среду Colab с GPU.")
print(torch.cuda.get_device_name())

def run_src(script, *args):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *args], cwd=repo, env=env, check=True)

if (repo / "docs" / "workflow-final500.svg").is_file():
    display(SVG(filename=str(repo / "docs" / "workflow-final500.svg")))

In [ ]:
if not (work / "split.json").is_file():
    if (data.is_relative_to("/content/drive") or work.is_relative_to("/content/drive")) and not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    run_src("prepare.py")
split = json.loads((work / "split.json").read_text())
assert split["trainval"] == 212_500 and split["test"] == 37_500
assert split["fold_sizes"] == [42_500] * 5
print("Разбиение подтверждено", split["fold_sizes"], "тест", split["test"])

## 2. Базовые экспертные прогнозы и поиск вариантов

V3 строит внешние прогнозы и внутренние cross-fit пакеты из 3 геоклассификаторов, прямой нейросети, локального поиска и статистик уверенности. V4 проверяет 50 способов коррекции и взвешивания. Публикации вдохновляют дополнительные 10 конфигураций. Здесь же 500 конфигураций проверяются на сокращённом разбиении внутри 85%: 60 000 fit, 20 000 calibration и 12 500 validation. Это скрининг для отбора, а не независимая оценка качества.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

v3 = work / "v3"
required = ["split.sha256", "cv-summary.json"] + [f"bundle-o{f}.npz" for f in range(5)]
required += [f"bundle-o{f}-i{i}.npz" for f in range(5) for i in range(3)]
if any(not (v3 / n).is_file() for n in required):
    run_src("run_v3.py", "cv")
assert all((v3/n).is_file() for n in required)
run_src("run_v4.py", "cv")
run_src("run_research.py")
if not (work / "screen500/summary.json").is_file():
    run_src("screen/screen.py")
screen = json.loads((work / "screen500/summary.json").read_text())
assert screen["n"] == 500 and len(screen["candidates"]) == 500
ranking = pd.DataFrame(screen["candidates"])
ranking["family"] = ranking.name.str.split("/").str[0]
display(ranking.groupby("family").agg(n=("name", "count"), best_RMSE=("rmse_km", "min"), median_RMSE=("rmse_km", "median")).round(2))
display(ranking.head(15).round(2))
ranking.boxplot(column="rmse_km", by="family", rot=15, figsize=(9,4))
plt.ylabel("Предварительный RMSE, км"); plt.suptitle(""); plt.show()

## 3. Пять фолдов и сравнение объединённых моделей

Победители каждого семейства и общий shortlist переобучаются на внешних 170 000 строках. Обучаемые веса получают внутренние OOF-прогнозы, а внешние 42 500 строк используются для оценки. V4 и метамодель в первом объединении добавляются фиксированными весами, чтобы не использовать невалидные вложенные признаки. Дополнительный стек использует честные вложенные блоки. Среди 300 базовых прогнозов отдельно ищем специалистов, чей остаточный сигнал дополняет V4, и проверяем эффект удаления этих специалистов. Все пять фолдов составляют этап подбора: предварительный отбор уже использовал часть разработки. RMSE объединяет квадраты расстояний, а не усредняет RMSE фолдов.

In [ ]:
cv_path = work / "full500/cv-summary.json"
def has_stage(key):
    return cv_path.is_file() and key in json.loads(cv_path.read_text())

if not cv_path.is_file():
    run_src("screen/full_search.py")
if not has_stage("strong_fusion"):
    run_src("screen/strong_fusion.py")
if not (work / "diversity500/results.json").is_file():
    run_src("screen/diversity_select.py")
if not has_stage("diversity_fusion"):
    run_src("screen/diversity_cv.py")
if not has_stage("tuned_fusion"):
    run_src("screen/tune_fusion.py")
cv = json.loads((work / "full500/cv-summary.json").read_text())
ranking = pd.DataFrame(cv["candidates"]).sort_values("cv_rmse_km")
assert cv["winner"] == ranking.iloc[0]["name"]
display(ranking[["name", "cv_mae_km", "cv_rmse_km", "fold_rmse_km"]].round(2))
ax = ranking.head(12).iloc[::-1].plot.barh(x="name", y="cv_rmse_km", figsize=(10,5), legend=False)
ax.set_xlabel("Пять фолдов: RMSE WGS84, км")
plt.show()
print("Победитель зафиксирован по CV:", cv["winner"])

Подробный подбор объединения: сетка из 285 конфигураций включает общий или раздельный коэффициент восток/север, шесть представлений контекста, восемь значений ridge, ограничение длины поправки, вес крупных ошибок и фиксированные коэффициенты. Ни один параметр не подбирается по тестовым меткам.

## 4. Одна финальная оценка

Только после определения победителя обучаем используемые эксперты на всех 212 500 строках. Для обучаемой комбинации прогнозы разработки получены вне обучающих фолдов. Сохраняем выбранную модель и предсказания до чтения истинных тестовых координат. В прошлых версиях уже публиковались агрегаты этого же теста, поэтому для независимого подтверждения прогресса нужен новый закрытый набор.

In [ ]:
run_src("screen/final_full.py")
result = json.loads((work / "full500/results.json").read_text())
assert result["winner"] == cv["winner"]
assert result["test"]["n"] == 37_500
display(pd.DataFrame([{"Модель": result["winner"], "MAE, км": result["test"]["mae_km"], "RMSE, км": result["test"]["rmse_km"]}]).round(2))
print("Артефакты:", work / "full500")